In [ ]:
# À lancer dans un terminal sur harold :
python - << 'EOF'
import nbformat as nbf

nb = nbf.v4.new_notebook()

cells = []

# ── Cell 0 : Markdown intro ────────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("""# Decision Trees for ED Patient Clustering — Scenario 2

This notebook fits and exports decision trees to interpret the HDBSCAN clustering results
from the Emergency Department (ED) patient resource-consumption study.

## Two types of trees are built, for each cluster solution (5 and 9 clusters):
- **Internal trees**: predict cluster membership from *consumption features* (what the patient received)
- **External trees**: predict cluster membership from *triage/clinical features* (what was known on arrival)

## Notebook structure
1. Imports & global config
2. Cluster labels & feature definitions
3. Ordinal encoding of clinical/triage variables
4. Helper functions
5. Internal decision trees (loop over both solutions)
6. External decision trees (loop over both solutions)
7. Output summary

---
### Fixes vs original scripts
- `EXTERNAL_FEATURES` was never defined → defined here (Section 2)
- `df` was overwritten inside the loop, losing `_ordinal` columns → `apply_ordinal_encoding()` now re-encodes after each `pd.read_csv`
- `LabelEncoder` usage was non-standard (`le.classes_ = ...`) → replaced by proper `le.fit()`
- `class_names_clusters` was sorted alphabetically → now uses `CLUSTER_ORDER` to preserve clinical ordering
- `FEATURE_RENAME` was defined but never applied in internal trees → internal trees use raw names (correct), external trees apply the rename
- `export_tree_image` used inconsistent dpi between PDF and PNG → unified via loop"""))

# ── Cell 1 : Imports ───────────────────────────────────────────────────────
cells.append(nbf.v4.new_code_cell("""# ── Standard library
import os

# ── Data manipulation
import pandas as pd
import numpy as np

# ── Sklearn — decision trees
from sklearn.tree import DecisionTreeClassifier, export_text, plot_tree, _tree
from sklearn.preprocessing import LabelEncoder

# ── Visualisation
import matplotlib
matplotlib.use('Agg')   # non-interactive backend: required on server / no display
import matplotlib.pyplot as plt
import dtreeviz

plt.style.use('default')
print('All imports OK.')"""))

# ── Cell 2 : Global config ─────────────────────────────────────────────────
cells.append(nbf.v4.new_code_cell("""# ── Base output directory
FULL_OUTPUT_DIR = 'Results/Regular_clustering/Full_dataset'

# ── Shared run settings
RUN_LABEL = 's2_balanced'
SCALER    = 'minmax'

# ── Clustering solutions to process
# Each entry maps a cluster count to its (mcs, ms) HDBSCAN parameters.
# mcs = min_cluster_size, ms = min_samples (absolute values used in filenames).
CLUSTERING_SOLUTIONS = {
    5: {'mcs': 2271, 'ms': 15},
    9: {'mcs': 3406, 'ms': 34},
}

print('Global config set.')
print(f'  Output base : {FULL_OUTPUT_DIR}')
print(f'  Solutions   : {list(CLUSTERING_SOLUTIONS.keys())} clusters')"""))

# ── Cell 3 : Cluster labels ────────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("---\n## 2. Cluster Labels & Feature Definitions"))

cells.append(nbf.v4.new_code_cell("""# ── Cluster label mappings ────────────────────────────────────────────────
# Maps HDBSCAN integer cluster IDs to human-readable clinical labels.
# -1 is always the outlier class.

CLUSTER_LABELS_5 = {
    1 : 'C1 — UHCD + Hospitalization + heavy workup',
    0 : 'C2 — Hospitalized + full workup',
    2 : 'C3 — Discharged + biology +/- ECG',
    4 : 'C4 — Isolated X-ray +/- CT',
    3 : 'C5 — Minimal consumption',
    -1: 'Outliers',
}
CLUSTER_ORDER_5 = [
    'C1 — UHCD + Hospitalization + heavy workup',
    'C2 — Hospitalized + full workup',
    'C3 — Discharged + biology +/- ECG',
    'C4 — Isolated X-ray +/- CT',
    'C5 — Minimal consumption',
    'Outliers',
]

CLUSTER_LABELS_9 = {
    0 : 'C1 — UHCD + Hospitalization + heavy workup',
    4 : 'C2 — Hospitalized + biology + imaging',
    5 : 'C3 — Hospitalized + biology',
    6 : 'C4 — Hospitalized + ECG + CT',
    7 : 'C5 — Hospitalized + ultrasound + biology',
    8 : 'C6 — Hospitalized + ECG',
    1 : 'C7 — Discharged + biology',
    3 : 'C8 — Isolated X-ray',
    2 : 'C9 — Minimal consumption',
    -1: 'Outliers',
}
CLUSTER_ORDER_9 = [
    'C1 — UHCD + Hospitalization + heavy workup',
    'C2 — Hospitalized + biology + imaging',
    'C3 — Hospitalized + biology',
    'C4 — Hospitalized + ECG + CT',
    'C5 — Hospitalized + ultrasound + biology',
    'C6 — Hospitalized + ECG',
    'C7 — Discharged + biology',
    'C8 — Isolated X-ray',
    'C9 — Minimal consumption',
    'Outliers',
]

# Attach to the solutions dict for easy looping
CLUSTERING_SOLUTIONS[5]['labels'] = CLUSTER_LABELS_5
CLUSTERING_SOLUTIONS[5]['order']  = CLUSTER_ORDER_5
CLUSTERING_SOLUTIONS[9]['labels'] = CLUSTER_LABELS_9
CLUSTERING_SOLUTIONS[9]['order']  = CLUSTER_ORDER_9

print('Cluster label mappings defined.')"""))

# ── Cell 4 : Feature definitions ──────────────────────────────────────────
cells.append(nbf.v4.new_code_cell("""# ── Internal features (resource consumption) ─────────────────────────────
# Variables used to BUILD the clusters. Used in internal trees.
IMAGING_COLS      = ['has_ultrasound', 'has_ct_scan', 'has_xray', 'has_mri']
BIO_COLS          = ['has_blood_test', 'has_culture', 'has_lumbar_puncture', 'has_blood_gas']
PROCEDURE_COLS    = ['had_ekg']
DISPOSITION_COLS  = ['hospitalization', 'observation_unit']
QUANTI_COLS       = ['imaging_exam_count', 'bio_exam_count']
INTERNAL_FEATURES = IMAGING_COLS + BIO_COLS + PROCEDURE_COLS + DISPOSITION_COLS + QUANTI_COLS

# ── External features (triage / clinical on arrival) ──────────────────────
# Variables known at triage before any resource is consumed.
# FIX: this list was missing from the original scripts (NameError at runtime).
EXTERNAL_FEATURES = [
    'bp_status_ordinal', 'hr_status_ordinal', 'temp_status_ordinal',
    'sat_status_ordinal', 'rr_status_ordinal', 'o2_flow_status_ordinal',
    'gcs_status_ordinal', 'cap_blood_sugar_status_ordinal',
    'pupils_status_ordinal', 'anisocoria_status_ordinal',
    'urine_dipstick_clean_status_ordinal', 'pain_status_ordinal',
    'breathalyzer_status_ordinal', 'hemocue_status_ordinal',
    'transport_ordinal', 'age', 'triage', 'sex',
]

# ── Human-readable names for external features (used in plot labels only) ─
# Applied only in external trees — internal trees use raw column names.
FEATURE_RENAME = {
    'bp_status_ordinal': 'Blood pressure', 'hr_status_ordinal': 'Heart rate',
    'temp_status_ordinal': 'Temperature', 'sat_status_ordinal': 'SpO2',
    'rr_status_ordinal': 'Respiratory rate', 'o2_flow_status_ordinal': 'O2 flow',
    'gcs_status_ordinal': 'GCS', 'cap_blood_sugar_status_ordinal': 'Blood sugar',
    'pupils_status_ordinal': 'Pupils size', 'anisocoria_status_ordinal': 'Anisocoria',
    'urine_dipstick_clean_status_ordinal': 'Urine dipstick', 'pain_status_ordinal': 'Pain',
    'breathalyzer_status_ordinal': 'Breathalyzer', 'hemocue_status_ordinal': 'Hemocue',
    'transport_ordinal': 'Transport mode', 'age': 'Age', 'triage': 'Triage level', 'sex': 'Sex',
}

print(f'Internal features : {len(INTERNAL_FEATURES)}')
print(f'External features : {len(EXTERNAL_FEATURES)}')"""))

# ── Cell 5 : Encoding dicts ────────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("""---
## 3. Ordinal Encoding of Clinical / Triage Variables

Raw clinical status columns (strings) → integers, so the tree treats them as ordered.
- `1` (or `0` for binary) = **not measured**
- Increasing integers = increasing clinical severity

`apply_ordinal_encoding()` is called inside each loop (Sections 5 & 6) on a freshly loaded
DataFrame, fixing the bug where `pd.read_csv` inside the loop overwrote `df` and silently
dropped all `_ordinal` columns."""))

cells.append(nbf.v4.new_code_cell("""STATUS_ENCODINGS = {
    # Blood pressure: hypotension < normotension < hypertension
    'bp_status': {'not_measured': 1, 'hypotension': 2, 'normotension': 3, 'hypertension': 4},
    # Heart rate: bradycardia < normocardia < tachycardia
    'hr_status': {'not_measured': 1, 'bradycardia': 2, 'normocardia': 3, 'tachycardia': 4},
    # Temperature: hypothermia < normothermia < hyperthermia
    'temp_status': {'not_measured': 1, 'hypothermia': 2, 'normothermia': 3, 'hyperthermia': 4},
    # SpO2: severe hypoxia < hypoxia < normal
    'sat_status': {'not_measured': 1, 'severe hypoxia': 2, 'hypoxia': 3, 'normal': 4},
    # Respiratory rate: bradypnea < normal < tachypnea
    'rr_status': {'not_measured': 1, 'bradypnea': 2, 'normal': 3, 'tachypnea': 4},
    # O2 flow — binary (0=not measured, 1=off, 2=on)
    'o2_flow_status': {'not_measured': 0, 'off': 1, 'on': 2},
    # GCS: severe < moderate impairment < normal
    'gcs_status': {'not_measured': 1, 'severe_impairment': 2, 'moderate_impairment': 3, 'normal': 4},
    # Blood sugar: hypoglycemia < normoglycemia < hyperglycemia
    'cap_blood_sugar_status': {'not_measured': 1, 'hypoglycemia': 2, 'normoglycemia': 3, 'hyperglycemia': 4},
    # Pupils: myosis < normal < mydriasis
    'pupils_status': {'not_measured': 1, 'myosis': 2, 'normal': 3, 'mydriasis': 4},
    # Anisocoria — binary (0=not measured, 1=no, 2=yes)
    'anisocoria_status': {'not_measured': 0, 'no': 1, 'yes': 2},
    # Urine dipstick — binary (0=not measured, 1=negative, 2=positive)
    'urine_dipstick_clean_status': {'not_measured': 0, 'negative': 1, 'positive': 2},
    # Pain: no pain < mild < moderate < severe
    'pain_status': {'not_measured': 1, 'no_pain': 2, 'mild_pain': 3, 'moderate_pain': 4, 'severe_pain': 5},
    # Breathalyzer — binary (0=not measured, 1=negative, 2=positive)
    'breathalyzer_status': {'not_measured': 0, 'negative': 1, 'positive': 2},
    # Hemocue: severe anemia < moderate < normal
    'hemocue_status': {'not_measured': 1, 'severe_anemia': 2, 'moderate_anemia': 3, 'normal': 4},
}

# Transport mode — ordered by medical urgency of arrival vehicle
TRANSPORT_ORDER = {
    'Unknown': 1, 'Personal': 2, 'Post medical advice': 3,
    'Ambulance': 4, 'Emergency services': 5,
}

print(f'Encoding dicts defined: {len(STATUS_ENCODINGS)} clinical variables + transport.')"""))

cells.append(nbf.v4.new_code_cell("""def apply_ordinal_encoding(df: pd.DataFrame) -> pd.DataFrame:
    \"\"\"
    Apply ordinal encoding to all clinical status columns in STATUS_ENCODINGS,
    and encode transport_grouped -> transport_ordinal.

    Returns a NEW DataFrame (input is not modified).
    Reports any unmapped values as warnings.
    \"\"\"
    df = df.copy()  # avoid mutating the caller's object

    for col, mapping in STATUS_ENCODINGS.items():
        if col not in df.columns:
            print(f'  ⚠️  Column \"{col}\" not found — skipped.')
            continue
        df[f'{col}_ordinal'] = df[col].map(mapping)
        n_nan = df[f'{col}_ordinal'].isna().sum()
        if n_nan > 0:
            print(f'  ⚠️  {col}: {n_nan} unmapped values (NaN introduced).')
        else:
            print(f'  ✅ {col}: encoded ({len(mapping)} categories)')

    if 'transport_grouped' in df.columns:
        df['transport_ordinal'] = df['transport_grouped'].map(TRANSPORT_ORDER)
        print(f'\\n  Transport ordinal distribution:')
        print(df['transport_ordinal'].value_counts().sort_index().to_string())
        print(f'  Transport NaN: {df[\"transport_ordinal\"].isna().sum()}')
    else:
        print('  ⚠️  \"transport_grouped\" not found — transport_ordinal not created.')

    return df

print('apply_ordinal_encoding() defined.')"""))

# ── Cell 6 : Helper functions ──────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("---\n## 4. Helper Functions"))

cells.append(nbf.v4.new_code_cell("""def export_tree_image(tree, feature_names, class_names, title, out_dir, filename):
    \"\"\"Save a decision tree visualisation as PDF and PNG.\"\"\"
    fig, ax = plt.subplots(figsize=(32, 14), facecolor='white')
    ax.set_facecolor('white')
    plot_tree(tree, feature_names=feature_names, class_names=class_names,
              filled=True, rounded=True, fontsize=8, ax=ax)
    ax.set_title(title, fontsize=14)
    plt.tight_layout()
    # FIX: unified via loop (original had dpi=150 for PDF, dpi=200 for PNG inconsistently)
    for ext, dpi in [('pdf', 150), ('png', 200)]:
        plt.savefig(os.path.join(out_dir, f'{filename}.{ext}'),
                    format=ext, dpi=dpi, bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'  Exported: {filename}.pdf / .png')


def export_feature_importance(tree, feature_names, title, out_dir, filename, color='steelblue'):
    \"\"\"Plot and save a horizontal bar chart of Gini feature importances (>0 only).\"\"\"
    importances = pd.Series(tree.feature_importances_, index=feature_names)
    importances = importances[importances > 0].sort_values(ascending=False)
    print(f'\\n  Feature importance — {title}:')
    print(importances.round(3).to_string())

    fig, ax = plt.subplots(figsize=(9, max(5, len(importances) * 0.4)), facecolor='white')
    ax.set_facecolor('white')
    importances.sort_values().plot(kind='barh', ax=ax, color=color, edgecolor='white')
    ax.axvline(importances.mean(), color='red', linestyle='--', alpha=0.6, label='Mean')
    ax.set_title(title, fontsize=13, fontweight='bold')
    ax.set_xlabel('Importance (Gini)')
    ax.legend()
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{filename}.png'), dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'  Exported: {filename}.png')


def get_target_rules(tree, feature_names, target_class):
    \"\"\"
    Extract all root-to-leaf paths predicting a given target class.
    Returns list of {'conditions': [...], 'n_samples': int, 'purity': float}.
    \"\"\"
    tree_ = tree.tree_
    classes = tree.classes_
    feat_names = [feature_names[i] if i != _tree.TREE_UNDEFINED else 'undefined'
                  for i in tree_.feature]
    rules = []

    def recurse(node, conditions):
        if tree_.feature[node] != _tree.TREE_UNDEFINED:
            name, threshold = feat_names[node], tree_.threshold[node]
            recurse(tree_.children_left[node],  conditions + [f'{name} <= {threshold:.2f}'])
            recurse(tree_.children_right[node], conditions + [f'{name} >  {threshold:.2f}'])
        else:
            predicted = classes[np.argmax(tree_.value[node])]
            n_samples = int(tree_.n_node_samples[node])
            purity    = float(np.max(tree_.value[node]) / n_samples)
            if predicted == target_class:
                rules.append({'conditions': conditions, 'n_samples': n_samples, 'purity': purity})

    recurse(0, [])
    return rules


def export_ordinal_legend(status_encodings, feature_rename, out_dir, filename, title):
    \"\"\"Save a formatted PNG table of the ordinal encoding legend.\"\"\"
    fig, ax = plt.subplots(figsize=(10, 12), facecolor='white')
    ax.axis('off')
    table_data = [['Variable', 'Value', 'Label']]
    for col, mapping in status_encodings.items():
        col_clean = feature_rename.get(f'{col}_ordinal', col.replace('_status','').replace('_',' '))
        for label, value in sorted(mapping.items(), key=lambda x: x[1]):
            table_data.append([col_clean, str(value), label.replace('_', ' ')])

    table = ax.table(cellText=table_data[1:], colLabels=table_data[0],
                     cellLoc='left', loc='center', colWidths=[0.35, 0.1, 0.35])
    table.auto_set_font_size(False)
    table.set_fontsize(9)
    table.scale(1, 1.4)

    for j in range(3):
        table[0, j].set_facecolor('#2c3e50')
        table[0, j].set_text_props(color='white', fontweight='bold')

    prev_var, colors, current_color = None, ['#f5f5f5', '#ffffff'], '#f5f5f5'
    for i in range(1, len(table_data)):
        var = table_data[i][0]
        if var != prev_var:
            current_color = colors[1] if current_color == colors[0] else colors[0]
            prev_var = var
        for j in range(3):
            table[i, j].set_facecolor(current_color)

    ax.set_title(title, fontsize=12, fontweight='bold', pad=20)
    plt.tight_layout()
    plt.savefig(os.path.join(out_dir, f'{filename}.png'), dpi=200,
                bbox_inches='tight', facecolor='white')
    plt.close()
    print(f'  Exported ordinal legend: {filename}.png')


print('All helper functions defined.')"""))

# ── Cell 7 : Internal trees ────────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("""---
## 5. Internal Decision Trees

**Features**: resource consumption variables (same 16 used to build the clusters).
**Goal**: verify cluster structure is mechanically recoverable from a shallow tree.

- Tree 1 — multiclass: predict cluster label (outliers excluded, max_depth=8)
- Tree 2 — binary: detect outliers vs clustered (full dataset, class_weight=balanced, max_depth=8)"""))

cells.append(nbf.v4.new_code_cell("""for n_clusters, cfg in CLUSTERING_SOLUTIONS.items():

    mcs, ms       = cfg['mcs'], cfg['ms']
    CLUSTER_ORDER = cfg['order']

    print('\\n' + '='*70)
    print(f'INTERNAL TREES — {n_clusters}-CLUSTER SOLUTION  (mcs={mcs}, ms={ms})')
    print('='*70)

    # ── Paths
    csv_path = os.path.join(FULL_OUTPUT_DIR, 'With_counts', SCALER, RUN_LABEL,
        f'final_mcs{mcs}_ms{ms}',
        f'clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv')
    out_dir = os.path.join(FULL_OUTPUT_DIR, 'With_counts', SCALER, RUN_LABEL,
        f'final_mcs{mcs}_ms{ms}', f'decision_tree_internal_{n_clusters}clusters')
    os.makedirs(out_dir, exist_ok=True)
    print(f'Input : {csv_path}')
    print(f'Output: {out_dir}')

    # ── Load fresh copy each iteration (no column pollution between solutions)
    df_raw   = pd.read_csv(csv_path, low_memory=False)
    df_model = df_raw[INTERNAL_FEATURES + ['cluster', 'cluster_label']].dropna()
    print(f'Loaded: {len(df_raw)} rows | After dropna: {len(df_model)} rows')

    # Ordered class list — no Outliers
    valid_classes = [c for c in CLUSTER_ORDER if c != 'Outliers']

    # ══════════════════════════════════════════════════════════════════════
    # TREE 1 — Multiclass: predict cluster from consumption features
    # ══════════════════════════════════════════════════════════════════════
    print('\\n── Tree 1: cluster structure (outliers excluded) ──')

    df_clusters = df_model[df_model['cluster'] != -1].copy()
    X1 = df_clusters[INTERNAL_FEATURES]
    y1 = df_clusters['cluster_label']
    print(f'  N = {len(df_clusters)}')
    print(y1.value_counts().to_string())

    tree1 = DecisionTreeClassifier(max_depth=8, min_samples_leaf=300, random_state=42)
    tree1.fit(X1, y1)

    # FIX: use CLUSTER_ORDER instead of sorted(y1.unique()) to get correct clinical ordering
    class_names1 = [c for c in valid_classes if c in y1.unique()]

    export_tree_image(tree1, INTERNAL_FEATURES, class_names1,
        title=f'Decision tree — Cluster structure ({n_clusters} clusters, internal)',
        out_dir=out_dir, filename=f'tree1_clusters_{n_clusters}clusters')
    export_feature_importance(tree1, INTERNAL_FEATURES,
        title=f'Feature importance — Cluster structure ({n_clusters} clusters)',
        out_dir=out_dir, filename=f'tree1_feature_importance_{n_clusters}clusters', color='steelblue')

    rules1 = export_text(tree1, feature_names=INTERNAL_FEATURES)
    with open(os.path.join(out_dir, f'tree1_rules_{n_clusters}clusters.txt'), 'w') as f:
        f.write(rules1)
    print(f'  Text rules saved.')

    # ══════════════════════════════════════════════════════════════════════
    # TREE 2 — Binary: detect outliers vs clustered patients
    # ══════════════════════════════════════════════════════════════════════
    print('\\n── Tree 2: outlier detection (full dataset) ──')

    df_outliers               = df_model.copy()
    df_outliers['is_outlier'] = (df_outliers['cluster'] == -1).astype(int)
    X2 = df_outliers[INTERNAL_FEATURES]
    y2 = df_outliers['is_outlier']
    print(f'  N = {len(X2)} | Outliers: {y2.sum()} ({y2.mean():.1%})')

    tree2 = DecisionTreeClassifier(max_depth=8, min_samples_leaf=40,
                                   class_weight='balanced', random_state=42)
    tree2.fit(X2, y2)

    export_tree_image(tree2, INTERNAL_FEATURES, ['Non-outlier', 'Outlier'],
        title=f'Decision tree — Outlier detection ({n_clusters} clusters, internal)',
        out_dir=out_dir, filename=f'tree2_outliers_{n_clusters}clusters')
    export_feature_importance(tree2, INTERNAL_FEATURES,
        title=f'Feature importance — Outlier detection ({n_clusters} clusters)',
        out_dir=out_dir, filename=f'tree2_feature_importance_{n_clusters}clusters', color='tomato')

    rules2 = export_text(tree2, feature_names=INTERNAL_FEATURES)
    with open(os.path.join(out_dir, f'tree2_rules_{n_clusters}clusters.txt'), 'w') as f:
        f.write(rules2)

    # Print leaf paths that predict outlier as majority class
    outlier_paths = get_target_rules(tree2, INTERNAL_FEATURES, target_class=1)
    print(f'\\n  Outlier leaf paths: {len(outlier_paths)}')
    if not outlier_paths:
        print('  No leaf predicts outliers as majority — try reducing max_depth or min_samples_leaf.')
    else:
        for r in outlier_paths:
            print(f'\\n    n={r["n_samples"]} (purity {r["purity"]:.0%})')
            for cond in r['conditions']:
                print(f'      {cond}')

    print(f'\\n✅ Internal trees done for {n_clusters}-cluster solution.')"""))

# ── Cell 8 : External trees ────────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("""---
## 6. External Decision Trees (dtreeviz)

**Features**: triage/clinical variables known at patient arrival.
**Goal**: assess how well cluster membership can be predicted from arrival data alone.

Shallower trees (max_depth=4) for interpretability — external features are noisier predictors."""))

cells.append(nbf.v4.new_code_cell("""for n_clusters, cfg in CLUSTERING_SOLUTIONS.items():

    mcs, ms       = cfg['mcs'], cfg['ms']
    CLUSTER_ORDER = cfg['order']

    print('\\n' + '='*70)
    print(f'EXTERNAL TREES — {n_clusters}-CLUSTER SOLUTION  (mcs={mcs}, ms={ms})')
    print('='*70)

    # ── Paths
    csv_path = os.path.join(FULL_OUTPUT_DIR, 'With_counts', SCALER, RUN_LABEL,
        f'final_mcs{mcs}_ms{ms}',
        f'clustering_{RUN_LABEL}_{SCALER}_mcs{mcs}_ms{ms}_labeled.csv')
    out_dir = os.path.join(FULL_OUTPUT_DIR, 'With_counts', SCALER, RUN_LABEL,
        f'final_mcs{mcs}_ms{ms}', f'decision_tree_external_{n_clusters}clusters')
    os.makedirs(out_dir, exist_ok=True)

    # ── Load & encode
    # FIX: re-apply ordinal encoding after each pd.read_csv.
    # In the original script, df was overwritten by read_csv inside the loop,
    # silently losing all _ordinal columns computed before the loop.
    df_raw = pd.read_csv(csv_path, low_memory=False)
    df     = apply_ordinal_encoding(df_raw)
    print(f'Loaded and encoded: {len(df)} rows')

    # ── Export encoding legend (reference table for the report)
    export_ordinal_legend(STATUS_ENCODINGS, FEATURE_RENAME,
        out_dir=out_dir, filename='ordinal_legend',
        title=f'Ordinal encoding legend — {n_clusters}-cluster solution')

    # ── Prepare model DataFrame
    cols_needed = [c for c in EXTERNAL_FEATURES + ['cluster_label'] if c in df.columns]
    df_model    = df[cols_needed].dropna()
    print(f'After dropna: {len(df_model)} rows')

    valid_classes = [c for c in CLUSTER_ORDER if c != 'Outliers']

    # ══════════════════════════════════════════════════════════════════════
    # TREE 1 — Multiclass: predict cluster from triage features
    # ══════════════════════════════════════════════════════════════════════
    print('\\n── Tree 1: cluster structure (outliers excluded) ──')

    df_clusters = df_model[df_model['cluster_label'] != 'Outliers'].copy()
    X1 = df_clusters[EXTERNAL_FEATURES].astype(float).rename(columns=FEATURE_RENAME)

    # FIX: use le.fit() instead of directly assigning le.classes_.
    # The original non-standard usage was fragile: if a class label was absent
    # in the CSV, transform() would raise a silent error or produce wrong results.
    classes_present = [c for c in valid_classes if c in df_clusters['cluster_label'].values]
    le1 = LabelEncoder()
    le1.fit(classes_present)
    y1           = le1.transform(df_clusters['cluster_label'])
    class_names1 = list(le1.classes_)

    print(f'  N = {len(X1)} | Classes: {class_names1}')

    tree1 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200,
                                   class_weight='balanced', random_state=42)
    tree1.fit(X1, y1)

    viz1 = dtreeviz.model(tree1, X_train=X1, y_train=y1,
                          feature_names=list(X1.columns), class_names=class_names1,
                          target_name='Cluster')
    viz1.view(fancy=True,  scale=1.5, orientation='LR').save(
        os.path.join(out_dir, f'tree1_clusters_{n_clusters}clusters.svg'))
    viz1.view(fancy=False, scale=1.2, orientation='TD').save(
        os.path.join(out_dir, f'tree1_clusters_{n_clusters}clusters_simple.svg'))
    print('  SVG exports done.')

    export_feature_importance(tree1, list(X1.columns),
        title=f'Feature importance — Tree 1 ({n_clusters} clusters, external)',
        out_dir=out_dir, filename=f'tree1_feature_importance_{n_clusters}clusters', color='steelblue')

    rules1 = export_text(tree1, feature_names=list(X1.columns))
    with open(os.path.join(out_dir, f'tree1_rules_{n_clusters}clusters.txt'), 'w') as f:
        f.write(rules1)
    print('  Text rules saved.')

    # ══════════════════════════════════════════════════════════════════════
    # TREE 2 — Binary: detect outliers from triage features
    # ══════════════════════════════════════════════════════════════════════
    print('\\n── Tree 2: outlier detection (full dataset) ──')

    X2 = df_model[EXTERNAL_FEATURES].astype(float).rename(columns=FEATURE_RENAME)
    y2 = (df_model['cluster_label'] == 'Outliers').astype(int)
    print(f'  N = {len(X2)} | Outliers: {y2.sum()} ({y2.mean():.1%})')

    tree2 = DecisionTreeClassifier(max_depth=4, min_samples_leaf=200,
                                   class_weight='balanced', random_state=42)
    tree2.fit(X2, y2)

    viz2 = dtreeviz.model(tree2, X_train=X2, y_train=y2,
                          feature_names=list(X2.columns), class_names=['Clustered', 'Outlier'],
                          target_name='Outlier')
    viz2.view(fancy=True,  scale=1.5, orientation='LR').save(
        os.path.join(out_dir, f'tree2_outliers_{n_clusters}clusters.svg'))
    viz2.view(fancy=False, scale=1.2, orientation='TD').save(
        os.path.join(out_dir, f'tree2_outliers_{n_clusters}clusters_simple.svg'))
    print('  SVG exports done.')

    export_feature_importance(tree2, list(X2.columns),
        title=f'Feature importance — Tree 2 ({n_clusters} clusters outliers, external)',
        out_dir=out_dir, filename=f'tree2_feature_importance_{n_clusters}clusters', color='tomato')

    rules2 = export_text(tree2, feature_names=list(X2.columns))
    with open(os.path.join(out_dir, f'tree2_rules_{n_clusters}clusters.txt'), 'w') as f:
        f.write(rules2)
    print('  Text rules saved.')

    print(f'\\n✅ External trees done for {n_clusters}-cluster solution.')"""))

# ── Cell 9 : Summary ───────────────────────────────────────────────────────
cells.append(nbf.v4.new_markdown_cell("""---
## 7. Output Summary

# Files generated for **each** cluster solution (N = 5 and 9):
""")